In [15]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [19]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/albani/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/albani/tugas4/
print("Berhasil diunggah ke HDFS: /user/albani/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
/bin/bash: line 1: hdfs: command not found
/bin/bash: line 1: hdfs: command not found
Berhasil diunggah ke HDFS: /user/albani/tugas4/transaksi_september_2026.csv


In [17]:
# Membaca dataset langsung dari HDFS
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

# 1. Menampilkan skema data
print("--- Skema Data ---")
df.printSchema()

# 2. Menampilkan jumlah total baris
total_baris = df.count()
print(f"Jumlah total baris: {total_baris}")

# 3. Menampilkan 10 baris pertama
print("\n--- 10 Baris Pertama ---")
df.show(10)

--- Skema Data ---
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah total baris: 1000

--- 10 Baris Pertama ---
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-2

In [23]:
import builtins
from pyspark.sql.functions import avg, col

# 1. Menghitung berapa banyak data kosong pada kolom rating
null_count = df.filter(col("rating").isNull()).count()
print(f"Jumlah data kosong pada kolom 'rating': {null_count}")

# 2. Menghitung nilai rata-rata rating (diisi pembulatan 2 desimal)
avg_rating = df.select(avg("rating")).first()[0]

# Menggunakan builtins.round agar menggunakan pembulatan Python biasa
avg_rating_rounded = builtins.round(float(avg_rating), 2)
print(f"Nilai rata-rata rating untuk pengisian data kosong: {avg_rating_rounded}")

# 3. Mengisi nilai null dengan nilai rata-rata rating
df_clean = df.na.fill({"rating": avg_rating_rounded})

# Memastikan data kosong sudah terisi
remaining_nulls = df_clean.filter(col("rating").isNull()).count()
print(
    f"Jumlah data kosong pada kolom 'rating' setelah ditangani: {remaining_nulls}"
)

Jumlah data kosong pada kolom 'rating': 204
Nilai rata-rata rating untuk pengisian data kosong: 4.15
Jumlah data kosong pada kolom 'rating' setelah ditangani: 0


In [24]:
from pyspark.sql.functions import when

# 1. Menambahkan kolom total_pendapatan
df_transformed = df_clean.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga_satuan")
)

# 2. Menambahkan kolom tier_transaksi
df_transformed = df_transformed.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"),
)

# Menampilkan hasil transformasi
df_transformed.select(
    "order_id",
    "unit_terjual",
    "harga_satuan",
    "total_pendapatan",
    "tier_transaksi",
).show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [25]:
from pyspark.sql.functions import count, round, sum

# 1. Kategori apa yang memiliki total_pendapatan tertinggi?
print("=== 1. Kategori dengan Total Pendapatan Tertinggi ===")
df_kategori = (
    df_transformed.groupBy("kategori")
    .agg(sum("total_pendapatan").alias("total_pendapatan_kategori"))
    .orderBy(col("total_pendapatan_kategori").desc())
)
df_kategori.show(1)

# 2. Kota mana dengan jumlah transaksi tier "Besar" terbanyak?
print("=== 2. Kota dengan Transaksi Tier 'Besar' Terbanyak ===")
df_kota_besar = (
    df_transformed.filter(col("tier_transaksi") == "Besar")
    .groupBy("kota")
    .agg(count("order_id").alias("jumlah_transaksi_besar"))
    .orderBy(col("jumlah_transaksi_besar").desc())
)
df_kota_besar.show(1)

# 3. Berapa rata-rata rating untuk masing-masing metode_pembayaran?
print("=== 3. Rata-rata Rating per Metode Pembayaran ===")
df_rating_metode = (
    df_transformed.groupBy("metode_pembayaran")
    .agg(round(avg("rating"), 2).alias("rata_rata_rating"))
    .orderBy(col("rata_rata_rating").desc())
)
df_rating_metode.show()

=== 1. Kategori dengan Total Pendapatan Tertinggi ===
+------------+-------------------------+
|    kategori|total_pendapatan_kategori|
+------------+-------------------------+
|Rumah Tangga|                138665000|
+------------+-------------------------+
only showing top 1 row

=== 2. Kota dengan Transaksi Tier 'Besar' Terbanyak ===
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

=== 3. Rata-rata Rating per Metode Pembayaran ===
+-----------------+----------------+
|metode_pembayaran|rata_rata_rating|
+-----------------+----------------+
|              COD|            4.17|
|    Transfer Bank|            4.16|
|         E-Wallet|            4.14|
|     Kartu Kredit|            4.12|
+-----------------+----------------+



In [32]:
# Simpan hasil olahan ke direktori /user/albani/tugas4/transaksi_processed_csv
output_hdfs = (
    "hdfs://localhost:9000/user/albani/tugas4/transaksi_processed_csv"
)

df_transformed.write.mode("overwrite").option("header", "true").csv(
    output_hdfs
)
print("Berhasil menyimpan data ke HDFS!")

Berhasil menyimpan data ke HDFS!
